In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
Q= torch.tensor([[2.,3.,8.,9.,0.],[1.,2.,8.,7.,1.],[9.,7.,3.,2.,9.]])
K=torch.tensor([[3.,6.,8.,9.,1.],[7.,3.,4.,1.,0.],[2.,8.,9.,1.,3.]])
V=torch.tensor([[4.,6.,8.,9.,1.],[7.,8.,9.,1.,7.],[2.,1.,3.,4.,8.]])

S=Q @ K.T
d= torch.softmax(S,dim=-1)
F= d @ V
print(F)
print(d)
print(S)

tensor([[4.0000, 6.0000, 8.0000, 9.0000, 1.0000],
        [4.0000, 6.0000, 8.0000, 9.0000, 1.0000],
        [2.0001, 1.0002, 3.0002, 4.0002, 7.9997]])
tensor([[1.0000e+00, 0.0000e+00, 8.7565e-27],
        [1.0000e+00, 3.0144e-40, 2.1151e-19],
        [4.5398e-05, 1.2664e-14, 9.9995e-01]])
tensor([[169.,  64., 109.],
        [143.,  52., 100.],
        [120.,  98., 130.]])


In [3]:
import torch
torch.manual_seed(0)
def attention(W_q,W_k,W_v,X):
    Q=X @ W_q
    K= X @ W_k
    V= X @ W_v
    score= Q @ K.T/K.shape[-1]**0.5
    weights = torch.softmax(score,dim=-1)
    out = weights @ V
    return score , weights, out, Q, K, V
W_q= torch.randn(512,5)/512**0.5
W_k=torch.randn(512,5)/512**0.5
W_v=torch.randn(512,5)/512**0.5
X=torch.randn(3,512)
score,weights,out,Q,K,V= attention(W_q,W_k,W_v,X)
print(score, weights,out,Q,K,V)
print(score.shape,weights.shape,out.shape)

print("\n\nseparate from above code , only for verification ")

manual= weights[0,0]*V[0] + weights[0,1]*V[1] + weights[0,2]*V[2]
print(manual)
print(out[0])

tensor([[ 0.2609, -0.5119, -0.1389],
        [-0.4669,  0.0088, -0.0457],
        [-0.6698, -1.9923,  0.4939]]) tensor([[0.4690, 0.2166, 0.3144],
        [0.2420, 0.3893, 0.3687],
        [0.2238, 0.0596, 0.7165]]) tensor([[-0.7092, -0.5292, -0.0450,  0.1246, -1.2361],
        [-0.8767, -0.4309, -0.3256, -0.0284, -0.9286],
        [-0.6828, -1.1100, -0.4563,  0.6516, -1.9347]]) tensor([[ 0.3410,  0.3670,  1.3952,  0.5903, -0.5516],
        [-1.5227,  0.2577,  0.1818, -0.6485, -0.9282],
        [ 0.6211, -0.5582, -0.4827,  0.9731, -1.3432]]) tensor([[ 0.4750,  1.0462,  0.3413, -0.0525,  0.7392],
        [-1.1366,  1.7724, -0.1870,  0.1245,  2.2118],
        [ 0.3836, -0.5095, -0.2775, -0.2682, -0.5276]]) tensor([[-0.4305, -0.3839,  0.6705,  0.1044, -1.3927],
        [-1.3126,  0.5268, -0.4903, -1.0508,  0.6790],
        [-0.7092, -1.4730, -0.8055,  0.9642, -2.3216]])
torch.Size([3, 3]) torch.Size([3, 3]) torch.Size([3, 5])


separate from above code , only for verification 
tensor([-0.7

In [4]:
import torch
torch.manual_seed(0)
def attention(W_q,W_k,W_v,X):
    Q= X @ W_q
    K= X @ W_k
    V= X @ W_v
    score= Q @ K.T/K.shape[-1]**0.5
    weights= torch.softmax(score, dim=-1)
    out = weights @ V
    return Q, K,V, score, weights, out

##setup----
d_model= 512
num_heads= 8 
d_head= d_model// num_heads

X= torch.randn(3,d_model)

##setting up weights for multi heads-----
W_qs= [torch.randn(d_model, d_head)/d_model**0.5 for _ in range(num_heads)]
W_ks= [torch.randn(d_model, d_head)/d_model**0.5 for _ in range(num_heads)]
W_vs= [torch.randn(d_model, d_head)/d_model**0.5 for _ in range(num_heads)]

##collecting head weights and outputs in a list-----
head_weights=[]
head_out= []
for h in range(num_heads):
    Q,K,V,score,weights,out= attention(W_qs[h],W_ks[h],W_vs[h],X)
    head_weights.append(weights)
    head_out.append(out)

concat=torch.concat(head_out,dim=-1)
W_o= torch.randn(d_model,d_model)/d_model**0.5

final= concat @ W_o

##print("query,key, value, score, weights, X in order:",Q,K,V,score, weights,X)
##print("\n\nfinalised output after multi head attention:",final)
print(len(head_out), head_out[0].shape)   # 8, [3, 64]
print(concat.shape)                               # [3, 512]
print(final.shape)

# 1. shapes first (head 7's values)
print(Q.shape, K.shape, V.shape)
print(score.shape, weights.shape, out.shape)

# 2. every row of weights must add to 1
print(weights.sum(dim=-1))

# 3. hand-check one score: word 0 looking at word 1
print(score[0, 1])
print((Q[0] @ K[1]) / 64**0.5)          # same number

# 4. hand-check one output row (a blend of V rows)
manual = weights[0,0]*V[0] + weights[0,1]*V[1] + weights[0,2]*V[2]
print(torch.allclose(manual, out[0], atol=1e-5))   # True

# 5. confirm the printout was head 7
print(torch.equal(weights, head_weights[7]))       # True

# 6. compare two heads
print(head_weights[0])
print(head_weights[1])

# 7. stapling check: head 3's notes sit in columns 192 to 255
print(torch.equal(concat[:, 3*64:4*64], head_out[3]))   # True

# 8. peek into big tensors
print(final[0, :5])                  # word 0, first 5 numbers
torch.set_printoptions(sci_mode=False, precision=4)
print(K)                             # now in normal decimals

8 torch.Size([3, 64])
torch.Size([3, 512])
torch.Size([3, 512])
torch.Size([3, 64]) torch.Size([3, 64]) torch.Size([3, 64])
torch.Size([3, 3]) torch.Size([3, 3]) torch.Size([3, 64])
tensor([1.0000, 1.0000, 1.0000])
tensor(1.4804)
tensor(1.4804)
True
True
tensor([[0.1685, 0.4169, 0.4146],
        [0.0497, 0.4942, 0.4561],
        [0.5304, 0.0242, 0.4454]])
tensor([[0.5193, 0.4158, 0.0649],
        [0.2793, 0.4709, 0.2498],
        [0.7212, 0.2076, 0.0712]])
True
tensor([-0.4171, -0.5321,  0.2669, -1.1797, -0.8802])
tensor([[-1.8837, -0.2153, -1.2504,  0.6034, -0.2005, -1.3808, -2.4033,  1.1002,
         -0.3253,  0.1554, -0.2366,  0.8369, -0.7167, -1.7659, -0.6512,  0.2400,
          0.5251,  0.6958,  0.5341, -2.8184, -0.4869, -0.1707,  0.2568, -1.8761,
          1.8120,  0.0765,  1.5204, -0.7580, -1.0735,  0.8765,  2.1882, -1.3688,
         -0.4316, -0.7911,  0.4899, -0.6200, -0.3792,  0.3110,  0.3275,  0.2615,
         -0.9415,  1.6005,  0.6381,  0.2188, -0.1578,  0.6713,  0.4991,  0.

In [5]:
import torch
import torch.nn as nn

def attention(Q, K, V, mask=None):
    # Paper equation (1): softmax(Q K^T / sqrt(d_k)) V
    score = Q @ K.transpose(-2, -1) / K.shape[-1] ** 0.5
    if mask is not None:
        score = score.masked_fill(mask == 0, float("-inf"))   # Section 3.2.3
    weights = torch.softmax(score, dim=-1)
    out = weights @ V
    return out, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        d_head = d_model // num_heads

        # same as your W_qs, W_ks, W_vs lists, but learnable
        self.W_qs = nn.ModuleList([nn.Linear(d_model, d_head, bias=False) for _ in range(num_heads)])
        self.W_ks = nn.ModuleList([nn.Linear(d_model, d_head, bias=False) for _ in range(num_heads)])
        self.W_vs = nn.ModuleList([nn.Linear(d_model, d_head, bias=False) for _ in range(num_heads)])
        self.W_o = nn.Linear(d_model, d_model, bias=False)       # your W_o

        self.last_weights = None   # only for inspecting, can be deleted

    def forward(self, q, k, v, mask=None):
        head_out, head_weights = [], []
        for h in range(self.num_heads):
            Q = self.W_qs[h](q)
            K = self.W_ks[h](k)
            V = self.W_vs[h](v)
            out, weights = attention(Q, K, V, mask)
            head_out.append(out)
            head_weights.append(weights)

        concat = torch.cat(head_out, dim=-1)                          # [batch, tokens, 512]
        self.last_weights = torch.stack(head_weights, dim=1).detach() # [batch, 8, tokens, tokens]
        return self.W_o(concat)                                       # one tensor only

In [6]:
x = torch.randn(3, 10, 512)
mha = MultiHeadAttention()
out = mha(x, x, x)
print(out.shape)                          # expect [3, 10, 512]
print(mha.last_weights.shape)             # expect [3, 8, 10, 10]
print(mha.last_weights.sum(dim=-1)[0, 0]) # expect ten 1s

mask = torch.tril(torch.ones(10, 10))     # token i can only see tokens up to i
out = mha(x, x, x, mask)
print(mha.last_weights[0, 0, 0])          # expect first number 1, the other nine 0

torch.Size([3, 10, 512])
torch.Size([3, 8, 10, 10])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [7]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length=5000, dropout=0.1):
        super().__init__()
        self.dropout= nn.Dropout(dropout)
        lam= torch.exp(torch.arange(0,d_model,2) * (-math.log(10000)/d_model))
        position= torch.arange(max_length).unsqueeze(1)
        pe= torch.zeros(max_length, d_model)
        pe[:,0::2]= torch.sin(lam * position)
        pe[:,1::2]= torch.cos(lam * position)
        self.register_buffer("pe",pe.unsqueeze(0))

    def forward(self,x):
        x= x + self.pe[:,: x.size(1)]
        return self.dropout(x)

In [8]:
d_model, max_len = 512, 100
pe = PositionalEncoding(d_model, max_len, dropout=0.0).pe[0]   # (max_len, d_model)

# Test 1: one entry against the paper's formula
pos, i = 7, 3
ang = torch.tensor(pos / 10000 ** (2 * i / d_model))
print(torch.isclose(pe[pos, 2*i],   torch.sin(ang), atol=1e-5))
print(torch.isclose(pe[pos, 2*i+1], torch.cos(ang), atol=1e-5))

# Test 2: the shift recipe, a*s + b*c, on every pair at once
k = 5
lam = 10000 ** (-torch.arange(0, d_model, 2) / d_model)
a, b = torch.cos(lam * k), torch.sin(lam * k)
s, c = pe[:-k, 0::2], pe[:-k, 1::2]                  # heights at t
new_s, new_c = a*s + b*c, a*c - b*s                  # recipe
print(torch.allclose(new_s, pe[k:, 0::2], atol=1e-4))
print(torch.allclose(new_c, pe[k:, 1::2], atol=1e-4))

tensor(True)
tensor(True)
True
True


In [9]:
import  torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model= 512, d_ff= 2048):
        super().__init__()
        self.expand= nn.Linear(d_model, d_ff)
        self.squeeze= nn.Linear(d_ff, d_model)

    def forward(self,x):
        x= self.expand(x)
        x= torch.relu(x)
        x= self.squeeze(x)
        return x


#Test
ffn= FeedForward()
x= torch.randn(3,512)
out= ffn(x)
print("shape of x\n",x.shape, "shape of out\n",out.shape)

shape of x
 torch.Size([3, 512]) shape of out
 torch.Size([3, 512])


In [10]:
#Embeddings code 
import torch
import torch.nn as nn
import math

class Tokenembedding(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.table = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.table.weight, mean=0.0, std=d_model ** -0.5)   # new line

    def forward(self, ids):
        return self.table(ids) * math.sqrt(self.d_model)

torch.manual_seed(0)
emb= Tokenembedding(512, 37000)
ids= torch.tensor([5, 912, 4])
out = emb(ids)
print(out.shape)
print(out)

torch.Size([3, 512])
tensor([[ 1.0718,  0.6388,  1.5033,  ...,  0.9824, -0.1536, -0.0509],
        [-0.9483,  0.0877,  0.0829,  ..., -0.4922, -0.4247, -1.8575],
        [ 0.6115,  0.1913, -0.2749,  ...,  1.2683,  1.5072, -0.5524]],
       grad_fn=<MulBackward0>)


In [11]:
# Byte Pair Encoding (BPE) tokenizer, written from scratch.
# Paper link: Section 5.1 says sentences were encoded with BPE,
# using ONE shared vocabulary for source and target language.
# So we train a single tokenizer on English + Hindi text together.

from collections import Counter

END = "</w>"  # marks "this is the end of a word"


class BPETokenizer:
    def __init__(self):
        self.merges = []          # list of pairs, in the order they were learned
        self.token_to_id = {}     # token string -> number
        self.id_to_token = {}     # number -> token string
        self.specials = ["<pad>", "<unk>", "<sos>", "<eos>"]

    # ---------- helper: break a word into starting pieces ----------
    def _word_to_symbols(self, word):
        return tuple(word) + (END,)   # "cat" -> ('c','a','t','</w>')

    # ---------- helper: count how often each neighbouring pair appears ----------
    def _count_pairs(self, word_freqs):
        pairs = Counter()
        for symbols, freq in word_freqs.items():
            for a, b in zip(symbols, symbols[1:]):
                pairs[(a, b)] += freq
        return pairs

    # ---------- helper: glue one chosen pair together inside every word ----------
    def _merge_pair(self, pair, word_freqs):
        new_freqs = {}
        for symbols, freq in word_freqs.items():
            out, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                    out.append(symbols[i] + symbols[i + 1])
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            new_freqs[tuple(out)] = new_freqs.get(tuple(out), 0) + freq
        return new_freqs

    # ---------- training: learn the merges ----------
    def train(self, texts, vocab_size):
        # 1) count every word in the corpus
        word_counts = Counter(w for line in texts for w in line.split())
        # 2) turn each word into characters + END
        word_freqs = {self._word_to_symbols(w): c for w, c in word_counts.items()}

        # 3) base vocabulary = every single character seen (+ END)
        base = {s for symbols in word_freqs for s in symbols}
        vocab = list(self.specials) + sorted(base)

        # 4) keep merging the most frequent pair until vocab is big enough
        while len(vocab) < vocab_size:
            pairs = self._count_pairs(word_freqs)
            if not pairs:
                break                      # nothing left to merge
            best = pairs.most_common(1)[0][0]
            word_freqs = self._merge_pair(best, word_freqs)
            self.merges.append(best)
            vocab.append(best[0] + best[1])

        self.token_to_id = {t: i for i, t in enumerate(vocab)}
        self.id_to_token = {i: t for t, i in self.token_to_id.items()}

    # ---------- split ONE word into tokens using the learned merges ----------
    def _tokenize_word(self, word):
        symbols = list(self._word_to_symbols(word))
        for a, b in self.merges:           # replay merges in learned order
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i + 1] == b:
                    symbols[i:i + 2] = [a + b]
                else:
                    i += 1
        return symbols

    # ---------- text -> list of ids ----------
    def encode(self, text, add_special=False):
        ids = []
        for word in text.split():
            for tok in self._tokenize_word(word):
                ids.append(self.token_to_id.get(tok, self.token_to_id["<unk>"]))
        if add_special:
            ids = [self.token_to_id["<sos>"]] + ids + [self.token_to_id["<eos>"]]
        return ids

    # ---------- list of ids -> text ----------
    def decode(self, ids):
        toks = [self.id_to_token[i] for i in ids if self.id_to_token[i] not in self.specials]
        return "".join(toks).replace(END, " ").strip()


# ---------------- quick test ----------------
if __name__ == "__main__":
    corpus = [
        "The Ministry of Defence signed a contract today",
        "The Indian Army conducted a joint exercise",
        "रक्षा मंत्रालय ने आज अनुबंध पर हस्ताक्षर किए",
        "भारतीय सेना ने संयुक्त अभ्यास किया",
    ] * 5   # repeated so pairs have counts

    tok = BPETokenizer()
    tok.train(corpus, vocab_size=40)

    print("vocab size:", len(tok.token_to_id))
    print("first 5 merges:", tok.merges[:5])

    for s in ["The Indian Army signed a contract", "भारतीय सेना ने अनुबंध किया"]:
        ids = tok.encode(s)
        print(s, "->", ids)
        print("decoded:", tok.decode(ids))

vocab size: 53
first 5 merges: []
The Indian Army signed a contract -> [9, 16, 13, 4, 7, 20, 12, 17, 10, 20, 4, 5, 22, 19, 27, 4, 23, 17, 15, 20, 13, 12, 4, 10, 4, 11, 21, 20, 24, 22, 10, 11, 24, 4]
decoded: The Indian Army signed a contract
भारतीय सेना ने अनुबंध किया -> [39, 47, 42, 34, 49, 41, 4, 45, 51, 36, 47, 4, 36, 51, 4, 29, 36, 50, 38, 28, 35, 4, 32, 48, 41, 47, 4]
decoded: भारतीय सेना ने अनुबंध किया


In [12]:
# Residual connection + Layer Normalization
# Paper, Section 3.1: the output of each sub-layer is
#     LayerNorm(x + Sublayer(x))
# Section 5.4: dropout is applied to the sub-layer output
# BEFORE it is added to x and normalized (base model: 0.1).

import torch
import torch.nn as nn


class LayerNorm(nn.Module):
    def __init__(self, d_model=512, eps=1e-6):
        super().__init__()
        self.gain = nn.Parameter(torch.ones(d_model))   # learned, starts at 1
        self.bias = nn.Parameter(torch.zeros(d_model))  # learned, starts at 0
        self.eps = eps                                  # tiny number, avoids dividing by 0

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)                  # average of the 512 numbers
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)   # how spread out they are
        x_norm = (x - mean) / torch.sqrt(var + self.eps)     # mean 0, spread 1
        return self.gain * x_norm + self.bias


class SublayerConnection(nn.Module):
    def __init__(self, d_model=512, dropout=0.1):
        super().__init__()
        self.norm = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        # sublayer is any function: attention, or the feed-forward network
        return self.norm(x + self.dropout(sublayer(x)))


# ---------------- quick test ----------------
if __name__ == "__main__":
    x = torch.randn(3, 10, 512)          # 3 sentences, 10 tokens each, 512 numbers per token
    block = SublayerConnection()
    out = block(x, lambda t: t * 2)      # stand-in for attention, just to test
    print(out.shape)                     # expect torch.Size([3, 10, 512])
    print(out.mean(dim=-1)[0, :3])       # expect values very close to 0
    print(out.std(dim=-1)[0, :3])   # expect values very close to 1

torch.Size([3, 10, 512])
tensor([-0.0000, -0.0000, -0.0000], grad_fn=<SliceBackward0>)
tensor([1.0010, 1.0010, 1.0010], grad_fn=<SliceBackward0>)


In [13]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, 8)
        self.feed_forward = FeedForward(d_model, 2048)
        self.sublayer1 = SublayerConnection(d_model, dropout)   # wraps attention
        self.sublayer2 = SublayerConnection(d_model, dropout)   # wraps feed-forward

    def forward(self, x, mask=None):
        x = self.sublayer1(x, lambda t: self.self_attn(t, t, t, mask))
        x = self.sublayer2(x, self.feed_forward)
        return x

In [14]:
# test for encoder layer
import copy

class Encoder(nn.Module):
    def __init__(self, layer, N=6):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

x = torch.randn(3, 10, 512)
encoder = Encoder(EncoderLayer(), N=6)
out = encoder(x)
print(out.shape)                  # expect [3, 10, 512]
print(out.mean(dim=-1)[0, :3])    # expect close to 0
print(out.std(dim=-1)[0, :3])     # expect close to 1

torch.Size([3, 10, 512])
tensor([ 0.0000, -0.0000, -0.0000], grad_fn=<SliceBackward0>)
tensor([1.0010, 1.0010, 1.0010], grad_fn=<SliceBackward0>)


In [15]:
#decoder layer
def subsequent_mask(size):
    # row i has 1s for columns 0..i, and 0s after: token i can't see the future
    return torch.tril(torch.ones(size, size))


class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, 8)    # masked, looks at decoder tokens
        self.cross_attn = MultiHeadAttention(d_model, 8)   # looks at encoder output
        self.feed_forward = FeedForward(d_model, 2048)
        self.sublayer1 = SublayerConnection(d_model, dropout)
        self.sublayer2 = SublayerConnection(d_model, dropout)
        self.sublayer3 = SublayerConnection(d_model, dropout)

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        x = self.sublayer1(x, lambda t: self.self_attn(t, t, t, tgt_mask))
        x = self.sublayer2(x, lambda t: self.cross_attn(t, memory, memory, src_mask))
        x = self.sublayer3(x, self.feed_forward)
        return x


class Decoder(nn.Module):
    def __init__(self, layer, N=6):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return x

In [16]:
#decoder layer check 
encoder = Encoder(EncoderLayer(), N=6)
decoder = Decoder(DecoderLayer(), N=6)
encoder.eval(); decoder.eval()      # turns dropout off, so results are repeatable

src = torch.randn(3, 10, 512)       # source sentence
tgt = torch.randn(3, 7, 512)        # target sentence so far
mask = subsequent_mask(7)

memory = encoder(src)
out1 = decoder(tgt, memory, mask)
print(out1.shape)                   # expect [3, 7, 512]

# causality check: change tokens 5 and 6, then compare tokens 0 to 4
tgt2 = tgt.clone()
tgt2[:, 5:] = torch.randn(3, 2, 512)
out2 = decoder(tgt2, memory, mask)
print(torch.allclose(out1[:, :5], out2[:, :5], atol=1e-5))   # expect True
print(torch.allclose(out1[:, 5:], out2[:, 5:], atol=1e-5))   # expect False

torch.Size([3, 7, 512])
True
False


In [17]:
#the output layer (linear + softmax)
class Generator(nn.Module):
    def __init__(self, d_model=512, vocab_size=1000):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.proj(x)          # raw scores (called logits)

In [18]:
#test on decoder output
gen = Generator(512, 1000)           # 1000 is a made-up vocab size for the test
logits = gen(out1)
probs = torch.softmax(logits, dim=-1)

print(logits.shape)                  # expect [3, 7, 1000]
print(probs.sum(dim=-1)[0])          # expect seven 1s

torch.Size([3, 7, 1000])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SelectBackward0>)


In [19]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, N=6, dropout=0.1, max_length=5000, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        # embedding + positional encoding, one pair for source, one for target
        self.src_embed = nn.Sequential(Tokenembedding(d_model, src_vocab),
                                       PositionalEncoding(d_model, max_length, dropout))
        self.tgt_embed = nn.Sequential(Tokenembedding(d_model, tgt_vocab),
                                       PositionalEncoding(d_model, max_length, dropout))
        self.encoder = Encoder(EncoderLayer(d_model, dropout), N)
        self.decoder = Decoder(DecoderLayer(d_model, dropout), N)
        self.generator = Generator(d_model, tgt_vocab)

    def make_src_mask(self, src):
        return (src != self.pad_id).unsqueeze(1)       # [batch, 1, src_len], True = real token

    def encode(self, src, src_mask=None):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, tgt, memory, src_mask=None):
        tgt_mask = subsequent_mask(tgt.size(1)).to(tgt.device)
        return self.decoder(self.tgt_embed(tgt), memory, tgt_mask, src_mask)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        memory = self.encode(src, src_mask)
        return self.generator(self.decode(tgt, memory, src_mask))

In [20]:
#model check
model = Transformer(src_vocab=1000, tgt_vocab=1200)

src = torch.randint(0, 1000, (3, 10))   # 3 source sentences, 10 token IDs each
tgt = torch.randint(0, 1200, (3, 7))    # 3 target sentences, 7 token IDs each

logits = model(src, tgt)
print(logits.shape)                                  # expect [3, 7, 1200]
print(sum(p.numel() for p in model.parameters()))    # expect 45843632

torch.Size([3, 7, 1200])
45843632


In [21]:
#the optimizer and learning-rate schedule 
def lr_factor(step, d_model=512, warmup=4000):
    step = max(step, 1)    # step 0 would give a divide-by-zero
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: lr_factor(s))

#check
for s in [1, 100, 4000, 16000, 100000]:
    print(s, f"{lr_factor(s):.2e}")

1 1.75e-07
100 1.75e-05
4000 6.99e-04
16000 3.49e-04
100000 1.40e-04


In [22]:
#the loss function
pad_id = 0   # placeholder: your tokenizer will decide the real pad ID later
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=pad_id)

In [23]:
#The shift by one
tgt_in  = tgt[:, :-1]    # drop the last token
tgt_out = tgt[:, 1:]     # drop the first token

#check
tgt = torch.randint(1, 1200, (3, 8))     # IDs start at 1, so none equal pad_id
src = torch.randint(1, 1000, (3, 10))
tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

logits = model(src, tgt_in)                                  # [3, 7, 1200]
loss = criterion(logits.reshape(-1, 1200), tgt_out.reshape(-1))
print(logits.shape)
print(loss.item())

#new check
for _ in range(5):
    tgt = torch.randint(1, 1200, (3, 8))
    src = torch.randint(1, 1000, (3, 10))
    logits = model(src, tgt[:, :-1])
    print(criterion(logits.reshape(-1, 1200), tgt[:, 1:].reshape(-1)).item())

torch.Size([3, 7, 1200])
7.289395809173584
7.021225929260254
7.212594985961914
7.4855451583862305
7.243417739868164
7.287534236907959


In [24]:
#the training loop
def train_step(model, src, tgt, optimizer, scheduler, criterion):
    model.train()                                   # dropout on
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]       # shift by one
    logits = model(src, tgt_in)                     # 1. forward pass
    loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))  # 2. loss
    optimizer.zero_grad()                           # 3. clear old gradients
    loss.backward()                                 # 4. gradients for every weight
    optimizer.step()                                # 5. update the weights
    scheduler.step()                                #    move to the next learning rate
    return loss.item()

In [25]:
#toy-training cell
def copy_accuracy(m):
    m.eval()
    with torch.no_grad():
        test = torch.randint(2, 11, (200, 10)); test[:, 0] = 1
        pred = m(test, test[:, :-1]).argmax(dim=-1)
        return (pred == test[:, 1:]).float().mean().item()

toy = Transformer(src_vocab=11, tgt_vocab=11, N=2)
toy_opt = torch.optim.Adam(toy.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
toy_sched = torch.optim.lr_scheduler.LambdaLR(toy_opt, lambda s: 0.3 * lr_factor(s, warmup=400))
toy_loss = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=0)

for step in range(1, 801):
    data = torch.randint(2, 11, (32, 10)); data[:, 0] = 1
    loss = train_step(toy, data, data, toy_opt, toy_sched, toy_loss)
    if step % 50 == 0:
        print(step, round(loss, 3), round(copy_accuracy(toy), 3))

50 1.847 0.366
100 0.743 0.946
150 0.564 0.999
200 0.558 0.992
250 0.566 0.976
300 0.553 0.995
350 0.578 0.989
400 0.593 0.989
450 0.577 0.982
500 0.604 0.986
550 0.582 0.992
600 0.643 0.994
650 0.561 0.999
700 0.582 0.997
750 0.551 1.0
800 0.525 0.998


In [26]:
#accuracy check for toy
toy.eval()
with torch.no_grad():
    test = torch.randint(2, 11, (200, 10)); test[:, 0] = 1
    pred = toy(test, test[:, :-1]).argmax(dim=-1)
    print((pred == test[:, 1:]).float().mean(dim=0))    # expect close to 1.0

tensor([1.0000, 1.0000, 0.9900, 1.0000, 0.9950, 1.0000, 1.0000, 1.0000, 1.0000])


In [27]:
#generating output (greedy decoding)
#The method (Section 3 of the paper: the decoder "generates an output sequence one element at a time"):

#Run the encoder once to get memory.
#Start the output with just the start symbol.
#Ask the decoder for the next token and take the most likely one (greedy).
#Append it to the output and repeat from step 3.

def greedy_decode(model, src, start_id, max_len):
    model.eval()
    with torch.no_grad():
        src_mask = model.make_src_mask(src)
        memory = model.encode(src, src_mask)                          # encoder runs once
        ys = torch.full((src.size(0), 1), start_id, dtype=torch.long, device=src.device)
        for _ in range(max_len - 1):
            out = model.decode(ys, memory, src_mask)                  # decoder on the output so far
            logits = model.generator(out[:, -1])                      # scores for the next token only
            next_id = logits.argmax(dim=-1, keepdim=True)             # most likely token
            ys = torch.cat([ys, next_id], dim=1)                      # append it and repeat
    return ys

In [28]:
#testing on toy model : greedy decoder
test = torch.randint(2, 11, (5, 10)); test[:, 0] = 1
gen = greedy_decode(toy, test, start_id=1, max_len=10)
print(test[0])
print(gen[0])
print((gen[:, 1:] == test[:, 1:]).float().mean())

tensor([1, 7, 9, 8, 6, 9, 2, 2, 9, 6])
tensor([1, 7, 9, 8, 6, 9, 2, 2, 9, 6])
tensor(1.)


In [29]:
#padding test
m = Transformer(src_vocab=11, tgt_vocab=11, N=2)
m.eval()

sent = torch.randint(2, 11, (1, 6))
a = torch.cat([sent, torch.zeros(1, 4, dtype=torch.long)], dim=1)   # padded to length 10
b = torch.cat([sent, torch.zeros(1, 8, dtype=torch.long)], dim=1)   # padded to length 14
tgt = torch.randint(2, 11, (1, 5))

with torch.no_grad():
    ma = m.encode(a, m.make_src_mask(a))
    mb = m.encode(b, m.make_src_mask(b))
    print(torch.allclose(ma[:, :6], mb[:, :6], atol=1e-5))     # expect True  (encoder mask)
    print(torch.allclose(m(a, tgt), m(b, tgt), atol=1e-5))     # expect True  (cross-attention mask too)
    na, nb = m.encode(a), m.encode(b)                          # no mask this time
    print(torch.allclose(na[:, :6], nb[:, :6], atol=1e-5))     # expect False

True
True
False
